In [1]:
import json
import openai
import requests

client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [2]:
# 함수 정의

def get_popular_movies():
    """인기 영화 목록을 가져온다."""
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    """영화 상세 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}").json()

def get_movie_credits(id):
    """영화 출연진/제작진 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}/credits").json()

# 함수 매핑
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [3]:
# AI에게 알려줄 도구(tools) 정의
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [4]:
# 대화 이력 (메모리)
messages = []
messages.append(
    {
        "role": "system",
        "content": "You are a movie expert agent. Answer questions about movies using the provided tools. Reply in the same language the user uses.",
    }
)

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    message = response.choices[0].message
    process_ai_response(message)

def process_ai_response(message):
    if message.tool_calls:
        # AI의 tool_calls 응답을 messages에 기록
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ],
            }
        )
        # 각 tool_call에 대해 실제 함수 실행
        for tc in message.tool_calls:
            fn_name = tc.function.name
            arguments = tc.function.arguments
            try:
                args = json.loads(arguments)
            except json.JSONDecodeError:
                args = {}
            print(f"Function Calling {fn_name}({args})")

            function_to_run = FUNCTION_MAP.get(fn_name)
            result = function_to_run(**args)

            # 실행 결과를 role: "tool"로 추가
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )
        # 도구 결과를 포함해서 AI 다시 호출
        call_ai()
    else:
        # 일반 텍스트 응답
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

In [5]:
print("Movie Agent (종료: q)")

while True:
    user_input = input("\nYou: ")
    if user_input in ("q", "quit"):
        break
    else:
        print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

Movie Agent (종료: q)
User: 안녕
AI: 안녕하세요! 어떤 영화에 대해 궁금한 점이 있으신가요?
User: 나 SF 영화 좋아해
AI: SF 영화는 정말 흥미진진하죠! 최근에 인기 있는 SF 영화를 알고 싶으신가요?
User: 프로젝트 헤일메리 봤어
Function Calling get_popular_movies({})
AI: "프로젝트 헤일메리"는 정말 흥미로운 SF 영화입니다! 이 영화는 과학 교사인 라이랜드 그레이스가 자신의 정체성과 우주에서의 임무를 잊고 우주선에서 깨어나는 이야기입니다. 태양의 소멸을 초래하는 신비로운 물질에 대한 수수께끼를 풀기 위해 그의 과학 지식과 비약적인 아이디어를 끌어내는 과정을 그립니다.

영화에 대한 더 많은 상세 정보가 필요하신가요? 아니면 다른 영화에 대해 궁금한 사항이 있으신가요?
User: 다른 영화 추천좀 해줘
Function Calling get_popular_movies({})
AI: 여기 최근에 인기 있는 몇 가지 SF 영화를 추천해드릴게요:

1. **프로젝트 헤일메리 (Project Hail Mary)**  
   - 개요: 과학 교사 라이랜드 그레이스가 우주선에서 깨어나 신비로운 물질의 수수께끼를 풀기 위해 과학 지식과 비약적인 아이디어를 활용하는 이야기입니다.
   - 개봉일: 2026년 3월 15일
   - 평점: 8.2

   ![프로젝트 헤일메리](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)

2. **아바타: 불과 재 (Avatar: Fire and Ash)**  
   - 개요: 제이크 숄리와 네이티리의 가족이 팬도라에서의 새로운 위협에 맞서 싸우는 이야기입니다.
   - 개봉일: 2025년 12월 17일
   - 평점: 7.4

   ![아바타: 불과 재](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **호퍼스 (Hoppers